# 10 — Avaliação v2 (leave-one-device-out)

Compara os modelos ResNet-18 **v1** (treino em PNG, JPEG 90 e misto) com os **v2** (compressão JPEG sorteada + aparelhos), nos mesmos testes.

**Ambiente:** GPU T4 de preferência. Na CPU também funciona (cerca de 1 hora).

Se a sessão cair, rode de novo desde a primeira célula: os modelos já avaliados são pulados.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, json, shutil, time
sys.path.append("/content/drive/MyDrive/tcc")
from src import config, modelos, treino

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(os.cpu_count())
print("Usando:", DISPOSITIVO)


@torch.no_grad()
def prever(modelo, X, tamanho_lote=64):
    """Probabilidade de cada crop ser IA (GPU ou CPU)."""
    modelo.eval()
    probs = []
    for i in range(0, len(X), tamanho_lote):
        xb = treino.preparar_lote(X[i:i + tamanho_lote], DISPOSITIVO)
        if DISPOSITIVO == "cuda":
            with torch.autocast("cuda", dtype=torch.float16):
                saida = modelo(xb)
        else:
            saida = modelo(xb)
        probs.append(torch.softmax(saida.float(), 1)[:, 1].cpu())
    return torch.cat(probs).numpy()


def carregar_pesos(caminho, tentativas=5):
    """Copia do Drive para o disco do Colab antes de carregar (evita erros de leitura do Drive)."""
    local = "/content/_pesos_tmp.pt"
    for t in range(tentativas):
        try:
            shutil.copy(caminho, local)
            return torch.load(local, map_location=DISPOSITIVO)
        except (OSError, SystemError, RuntimeError) as e:
            print(f"  Falha ao ler {caminho} ({e}); tentativa {t + 1}/{tentativas}")
            time.sleep(10)
    raise RuntimeError(f"Não foi possível ler {caminho}")

Mounted at /content/drive
Usando: cuda


## Conjuntos de teste
Os testes da base (`original`, `jpeg90`, `firefly`) são os mesmos da Etapa 1. As fotos dos aparelhos são as versões **nativas** (sem recompressão).

In [3]:
APARELHOS = ["samsung_a56", "samsung_s23fe", "canon_powershot_sx510hs"]

FONTES = {
    "original": f"{config.ELA_DIR}/etapa1/teste_original",
    "jpeg90":   f"{config.ELA_DIR}/etapa1/teste_jpeg90",
    "firefly":  f"{config.ELA_DIR}/etapa1/teste_firefly",
    **{f"disp_{a}": f"{config.ELA_DIR}/v2/disp_nativo_{a}" for a in APARELHOS},
}

dados = {}
for chave, base in FONTES.items():
    dados[(chave, "rotulos")] = pd.read_csv(f"{base}_rotulos.csv")
    for entrada in ["ela", "rgb"]:
        dados[(chave, entrada)] = torch.from_numpy(np.load(f"{base}_{entrada}.npy"))
    print(f"{chave:35s} {len(dados[(chave, 'rotulos')])} crops")

original                            1200 crops
jpeg90                              1200 crops
firefly                             600 crops
disp_samsung_a56                    396 crops
disp_samsung_s23fe                  176 crops
disp_canon_powershot_sx510hs        260 crops


In [4]:
def por_imagem(rotulos, probs):
    """Junta os 4 crops de cada imagem pela média da probabilidade de ser IA."""
    d = rotulos[["id", "classe", "rotulo", "aparelho_ou_gerador"]].copy()
    d["prob_ia"] = probs
    return d.groupby(["id", "classe", "rotulo", "aparelho_ou_gerador"], as_index=False)["prob_ia"].mean()


def calcular_metricas(img):
    y = img["rotulo"].values
    p = img["prob_ia"].values
    pred = (p >= 0.5).astype(int)
    vn, fp, fn, vp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    tem_duas = len(set(y)) > 1
    acerto_reais = vn / (vn + fp) if (vn + fp) else np.nan
    deteccao_ia = vp / (vp + fn) if (vp + fn) else np.nan
    return {
        "acuracia": accuracy_score(y, pred),
        "acc_balanceada": np.nanmean([acerto_reais, deteccao_ia]) if tem_duas else np.nan,
        "acerto_reais": acerto_reais, "deteccao_ia": deteccao_ia,
        "f1": f1_score(y, pred, zero_division=0) if tem_duas else np.nan,
        "auc": roc_auc_score(y, p) if tem_duas else np.nan,
        "vp": vp, "fn": fn, "fp": fp, "vn": vn,
    }

## Avaliar os modelos v1 e v2
Os v2 são testados só no aparelho deixado de fora na rodada deles. Os v1 são testados nos três aparelhos (nunca viram nenhum).

In [5]:
SAIDA = f"{config.RESULTADOS}/v2"
os.makedirs(f"{SAIDA}/predicoes", exist_ok=True)
ARQ = f"{SAIDA}/metricas.csv"


def eh_alvo(nome):
    if "amostra" in nome or not nome.startswith("resnet18_"):
        return False
    return "_v2_" in nome or any(f"_{e}_seed" in nome for e in ["png", "jpeg90", "misto"])


rodadas = sorted(d for d in os.listdir(config.MODELOS)
                 if eh_alvo(d) and os.path.exists(f"{config.MODELOS}/{d}/melhor.pt"))
print(len(rodadas), "modelos (esperado: 36)")

linhas = pd.read_csv(ARQ).to_dict("records") if os.path.exists(ARQ) else []
feitas = {l["nome"] for l in linhas}

for nome in tqdm(rodadas):
    if nome in feitas:
        continue
    cfg = json.load(open(f"{config.MODELOS}/{nome}/config.json"))
    modelo = modelos.criar_modelo(cfg["modelo"])
    modelo.load_state_dict(carregar_pesos(f"{config.MODELOS}/{nome}/melhor.pt"))
    modelo.to(DISPOSITIVO).eval()

    if "aparelho_teste" in cfg:
        versao, aparelhos_teste = "v2", [cfg["aparelho_teste"]]
    else:
        versao, aparelhos_teste = f"v1_{cfg['experimento']}", APARELHOS

    imgs = {}
    for chave in ["original", "jpeg90", "firefly"] + [f"disp_{a}" for a in aparelhos_teste]:
        probs = prever(modelo, dados[(chave, cfg["entrada"])])
        imgs[chave] = por_imagem(dados[(chave, "rotulos")], probs)
        imgs[chave].to_csv(f"{SAIDA}/predicoes/{nome}_{chave}.csv", index=False)

    ia_jpeg = imgs["jpeg90"][imgs["jpeg90"]["rotulo"] == 1]
    for a in aparelhos_teste:
        imgs[f"disp_{a}_vs_ia"] = pd.concat([imgs[f"disp_{a}"], ia_jpeg], ignore_index=True)

    for teste, img in imgs.items():
        linhas.append({"nome": nome, "entrada": cfg["entrada"], "versao": versao,
                       "seed": cfg["seed"], "teste": teste, "n_imagens": len(img),
                       **calcular_metricas(img)})

    pd.DataFrame(linhas).to_csv(ARQ, index=False)
    del modelo
    if DISPOSITIVO == "cuda":
        torch.cuda.empty_cache()

print("Avaliação concluída.")

36 modelos (esperado: 36)


  0%|          | 0/36 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth



  0%|          | 0.00/44.7M [00:00<?, ?B/s]
 29%|██▊       | 12.8M/44.7M [00:00<00:00, 133MB/s]
100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s]
100%|██████████| 36/36 [02:53<00:00,  4.82s/it]

Avaliação concluída.


## Tabela comparativa (média ± desvio das 3 seeds)

| Coluna | Métrica |
|---|---|
| `original`, `jpeg90` | Acurácia |
| `firefly` | Taxa de detecção de IA |
| `aparelho_novo` | % de fotos do aparelho nunca visto reconhecidas como reais |
| `aparelho_novo_vs_ia` | Acurácia balanceada (fotos do aparelho × IA em JPEG 90) |

In [6]:
m = pd.read_csv(ARQ)
m["teste_geral"] = m["teste"]
for a in APARELHOS:
    m.loc[m["teste"] == f"disp_{a}", "teste_geral"] = "aparelho_novo"
    m.loc[m["teste"] == f"disp_{a}_vs_ia", "teste_geral"] = "aparelho_novo_vs_ia"

METRICA = {"original": "acuracia", "jpeg90": "acuracia", "firefly": "deteccao_ia",
           "aparelho_novo": "acerto_reais", "aparelho_novo_vs_ia": "acc_balanceada"}
m["valor"] = [row[METRICA[row["teste_geral"]]] for _, row in m.iterrows()]

def media_desvio(s):
    s = s.dropna()
    return f"{100 * s.mean():.1f} ± {100 * s.std():.1f}" if len(s) else ""

tab = (m.groupby(["entrada", "versao", "teste_geral"])["valor"].apply(media_desvio)
       .unstack("teste_geral")[list(METRICA)])
tab.to_csv(f"{SAIDA}/tabela_comparativa.csv")
print("TABELA COMPARATIVA (%)")
display(tab)

por_aparelho = (m[m["teste"].isin([f"disp_{a}" for a in APARELHOS])]
                .groupby(["entrada", "versao", "teste"])["acerto_reais"].apply(media_desvio)
                .unstack("teste"))
por_aparelho.to_csv(f"{SAIDA}/acerto_por_aparelho.csv")
print("\nFOTOS RECONHECIDAS COMO REAIS, POR APARELHO (%)")
display(por_aparelho)

TABELA COMPARATIVA (%)


teste_geral           original      jpeg90      firefly aparelho_novo  \
entrada versao                                                          
ela     v1_jpeg90   53.6 ± 3.7  99.1 ± 0.2   20.9 ± 8.7     0.3 ± 0.6   
        v1_misto    99.6 ± 0.2  98.7 ± 0.6   29.3 ± 2.9     0.2 ± 0.4   
        v1_png      99.3 ± 0.0  67.8 ± 2.0   25.6 ± 6.9     1.4 ± 2.1   
        v2          82.3 ± 4.0  91.0 ± 4.8  61.9 ± 17.2   12.4 ± 14.7   
rgb     v1_jpeg90   99.8 ± 0.4  99.9 ± 0.2   65.1 ± 3.4   20.1 ± 10.0   
        v1_misto    99.7 ± 0.6  99.6 ± 0.5  64.9 ± 13.0   16.0 ± 11.3   
        v1_png     100.0 ± 0.0  94.4 ± 2.5   89.6 ± 3.4     3.3 ± 2.6   
        v2          99.6 ± 0.4  99.5 ± 0.4   38.1 ± 9.5   79.4 ± 14.0   

teste_geral       aparelho_novo_vs_ia  
entrada versao                         
ela     v1_jpeg90          49.9 ± 0.4  
        v1_misto           49.9 ± 0.4  
        v1_png             23.4 ± 3.4  
        v2                 47.9 ± 8.7  
rgb     v1_jpeg90          60.0 ± 5.0  
        v1_misto           57.8 ± 5.4  
        v1_png             51.7 ± 1.3  
        v2                 89.2 ± 6.9


FOTOS RECONHECIDAS COMO REAIS, POR APARELHO (%)


teste             disp_canon_powershot_sx510hs disp_samsung_a56  \
entrada versao                                                    
ela     v1_jpeg90                    0.5 ± 0.9        0.3 ± 0.6   
        v1_misto                     0.0 ± 0.0        0.7 ± 0.6   
        v1_png                       0.5 ± 0.9        3.7 ± 2.1   
        v2                           5.1 ± 3.2      25.3 ± 18.4   
rgb     v1_jpeg90                   29.7 ± 2.4       22.9 ± 3.1   
        v1_misto                   21.5 ± 11.1       20.5 ± 9.4   
        v1_png                       5.6 ± 0.9        4.4 ± 0.6   
        v2                          62.6 ± 4.9      86.9 ± 10.3   

teste             disp_samsung_s23fe  
entrada versao                        
ela     v1_jpeg90          0.0 ± 0.0  
        v1_misto           0.0 ± 0.0  
        v1_png             0.0 ± 0.0  
        v2                6.8 ± 11.8  
rgb     v1_jpeg90          7.6 ± 1.3  
        v1_misto           6.1 ± 8.6  
        v1_png             0.0 ± 0.0  
        v2                88.6 ± 4.5

## Teste do atalho "sem JPEG = real"

Os modelos v2 nunca viram imagens sem compressão JPEG no treino. Esta tabela separa o teste `original` (PNG, **nunca comprimido**) por classe:

- se o v2 tiver **detecção de IA baixa** no `original`, ele aprendeu "sem JPEG = real" e o acerto nas fotos HEIC não é confiável;
- se a **detecção de IA continuar alta**, o acerto no HEIC é mais convincente.

In [7]:
atalho = (m[m["teste"].isin(["original", "jpeg90"])]
          .groupby(["entrada", "versao", "teste"])[["acerto_reais", "deteccao_ia"]]
          .agg(lambda s: f"{100 * s.mean():.1f} ± {100 * s.std():.1f}")
          .unstack("teste"))
atalho.to_csv(f"{SAIDA}/teste_atalho_png.csv")
display(atalho)

acerto_reais               deteccao_ia             
teste                   jpeg90     original       jpeg90     original
entrada versao                                                       
ela     v1_jpeg90   98.7 ± 0.7  18.4 ± 13.0   99.6 ± 0.4   88.7 ± 9.3
        v1_misto    97.8 ± 1.7   99.6 ± 0.4   99.6 ± 0.8   99.6 ± 0.4
        v1_png      90.2 ± 4.5   99.3 ± 0.0   45.3 ± 8.4   99.3 ± 0.0
        v2          98.7 ± 1.1   86.4 ± 9.6   83.3 ± 9.5  78.3 ± 12.1
rgb     v1_jpeg90   99.8 ± 0.4  100.0 ± 0.0  100.0 ± 0.0   99.6 ± 0.8
        v1_misto    99.6 ± 0.4   99.6 ± 0.8   99.6 ± 0.8   99.8 ± 0.4
        v1_png      88.9 ± 4.9  100.0 ± 0.0  100.0 ± 0.0  100.0 ± 0.0
        v2          99.9 ± 0.2   99.9 ± 0.3   99.1 ± 0.9   99.3 ± 0.7